In [ ]:
import pandas as pd
import geopandas as gpd
import os, re
from shapely.geometry import Point, Polygon, MultiPolygon
import numpy as np

import numpy as np
from typing import Literal

from datetime import datetime, time
import geopandas as gpd
import xarray as xr


import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

import rioxarray

import requests
import json
from shapely.geometry import shape

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter

import networkx as nx



In [ ]:
# Construct power grid network

def get_start_end_coordinates(geom):
    start_points = []
    end_points = []
    for line in geom.geoms:  # Access individual LineString geometries within the MultiLineString
        start_points.append(line.coords[0])   # Get the first coordinate of each LineString
        end_points.append(line.coords[-1])    # Get the last coordinate of each LineString
    return start_points, end_points




In [ ]:
file_path_powergrid = "/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/physical_grid_data/U.S._Electric_Power_Transmission_Lines.geojson"
gdf_powergrid = gpd.read_file(file_path_powergrid)



# Convert timeframes to folium-friendly types
gdf_powergrid['SOURCEDATE'] = pd.to_datetime(gdf_powergrid['SOURCEDATE']).dt.strftime('%Y-%m-%dT%H:%M:%S')
gdf_powergrid['VAL_DATE'] = pd.to_datetime(gdf_powergrid['VAL_DATE']).dt.strftime('%Y-%m-%dT%H:%M:%S')


# Drop columns with 'NOT AVAILABLE' as substation value
# gdf[ (gdf['SUB_1']=='NOT AVAILABLE') | (gdf['SUB_2']=='NOT AVAILABLE')]
condition = ( (gdf_powergrid['SUB_1']=='NOT AVAILABLE') | (gdf_powergrid['SUB_2']=='NOT AVAILABLE') )
gdf_powergrid = gdf_powergrid[~condition]
# gdf

# Drop columns with 'NONE' as substation value
condition = ( (gdf_powergrid['SUB_1']=='NONE') | (gdf_powergrid['SUB_2']=='NONE') )
gdf_powergrid = gdf_powergrid[~condition]
gdf_powergrid



In [ ]:
powergrid_map = gdf_powergrid.plot(figsize=(20, 16),linewidth=0.5)


In [ ]:
file_path_states = "/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/location_data/Census_Bureau_Data/tl_2023_us_state/tl_2023_us_state.shp"
gdf_states = gpd.read_file(file_path_states)
gdf_states = gdf_states.to_crs(gdf_powergrid.crs)

In [ ]:
powergrid_plot = gdf_powergrid
# Plot power grid transmission lines
powergrid_plot.plot(ax=ax, color='black', linewidth=0.5, label='Transmission Lines', zorder=2)

# Overlay state boundaries on top with different styling
gdf_states.plot(ax=ax, edgecolor='red', facecolor='none', linewidth=2.5, label='State Boundaries', zorder=4)

ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('Power Grid Transmission Lines with Grid Tiles and State Boundaries', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Apply the start and end points function to the MultiLineString geometry
gdf_powergrid['start_points'], gdf_powergrid['end_points'] = zip(*gdf_powergrid['geometry'].apply(lambda geom: get_start_end_coordinates(geom)))


In [ ]:
from shapely.geometry import Point, LineString

# Create a NetworkX graph
G = nx.Graph()

# Add edges to the graph
for idx, row in gdf_powergrid.iterrows():
    start = tuple(row['start_points'])
    end = tuple(row['end_points'])
    start_id = f"{start[0][0]}_{start[0][1]}"
    end_id = f"{end[0][0]}_{end[0][1]}"
    G.add_edge(start_id, end_id, geometry=LineString([start[0], end[0]]))


# Calculate degree centrality
centrality = nx.degree_centrality(G)

# Get the top X nodes based on degree centrality
num_nodes = 1000
top_X_nodes = sorted(centrality, key=centrality.get, reverse=True)[:num_nodes]

# Get the subgraph of the top X nodes
subgraph = G.subgraph(top_X_nodes)

# Create a GeoDataFrame of the edges for plotting
edges = []
for u, v, data in subgraph.edges(data=True):
    start_coords = tuple(map(float, u.split('_')))
    end_coords = tuple(map(float, v.split('_')))
    edges.append(LineString([start_coords, end_coords]))

edges_gdf = gpd.GeoDataFrame(geometry=edges)

# Create a GeoDataFrame of the nodes for plotting
nodes = [Point(tuple(map(float, node.split('_')))) for node in top_X_nodes]
nodes_gdf = gpd.GeoDataFrame(geometry=nodes)



In [ ]:
# Build complete node and edge GeoDataFrames (WGS84) for the entire network
# Nodes: parse node IDs (lon_lat) to Point geometries
node_geoms = []
node_ids = []
for node_id in G.nodes():
    try:
        lon_str, lat_str = node_id.split('_')
        lon, lat = float(lon_str), float(lat_str)
        node_geoms.append(Point(lon, lat))
        node_ids.append(node_id)
    except Exception as e:
        print(f"Warning: could not parse node {node_id}: {e}")
        
nodes_gdf = gpd.GeoDataFrame({
    'node_id': node_ids,
    'degree': [G.degree(n) for n in node_ids]
}, geometry=node_geoms, crs='EPSG:4326')

# Edges: reconstruct LineString from node IDs
edge_geoms = []
edge_ids = []
edge_start = []
edge_end = []
for u, v in G.edges():
    try:
        u_lon, u_lat = map(float, u.split('_'))
        v_lon, v_lat = map(float, v.split('_'))
        edge_geoms.append(LineString([(u_lon, u_lat), (v_lon, v_lat)]))
        edge_ids.append(f"{u}|{v}")
        edge_start.append(u)
        edge_end.append(v)
    except Exception as e:
        print(f"Warning: could not parse edge ({u}, {v}): {e}")

edges_gdf_full = gpd.GeoDataFrame({
    'edge_id': edge_ids,
    'start_node': edge_start,
    'end_node': edge_end,
}, geometry=edge_geoms, crs='EPSG:4326')

print(f"Built GeoDataFrames: {len(nodes_gdf)} nodes, {len(edges_gdf_full)} edges")

In [ ]:
# Enhanced network visualization: nodes + edges + AOI + hazard tiles
fig, ax = plt.subplots(figsize=(16, 12))

# Plot AOI boundary
gdf_states.to_crs('EPSG:4326').boundary.plot(ax=ax, color='black', linewidth=2, zorder=5, label='AOI')

# Plot transmission line edges (blue)
edges_gdf_full.plot(ax=ax, color='steelblue', linewidth=0.8, alpha=0.7, zorder=3, label='Transmission Lines')

# Plot substation nodes colored by degree (connectivity)
nodes_gdf.plot(
    ax=ax,
    column='degree',
    cmap='Reds',#'YlOrRd',
    markersize=15,
    alpha=0.85,
    edgecolor='black',
    linewidth=0.5,
    legend=True,
    zorder=4,
    legend_kwds={'label': 'Node Degree (# connections)', 'shrink': 0.7}
)

ax.set_title('Power Grid Network: Substations (nodes) & Transmission Lines (edges)', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(loc='upper right', fontsize=10)

# Restrict map extent to CONUS
ax.set_xlim(-125, -66.5)
ax.set_ylim(24, 49.5)

plt.tight_layout()
plt.show()

print(f"Node degree range: {nodes_gdf['degree'].min()} to {nodes_gdf['degree'].max()}")

In [ ]:
# Simplified network view: highest-degree nodes only, sized by relative centrality

# Parameters
top_n = 150  # number of highest-degree nodes to display
conus_bounds = {"lon_min": -125, "lon_max": -66.5, "lat_min": 24, "lat_max": 49.5}

# Ensure consistent CRS and keep only CONUS nodes
nodes_plot = nodes_gdf.to_crs("EPSG:4326").copy()
nodes_plot = nodes_plot[
    (nodes_plot.geometry.x >= conus_bounds["lon_min"]) &
    (nodes_plot.geometry.x <= conus_bounds["lon_max"]) &
    (nodes_plot.geometry.y >= conus_bounds["lat_min"]) &
    (nodes_plot.geometry.y <= conus_bounds["lat_max"])
].copy()

# Keep highest-degree nodes
top_nodes = nodes_plot.nlargest(min(top_n, len(nodes_plot)), "degree").copy()

# Relative centrality from degree (0 to 1)
max_degree = nodes_plot["degree"].max() if len(nodes_plot) > 0 else 1
top_nodes["relative_centrality"] = top_nodes["degree"] / max_degree

# Dot size scales with centrality
top_nodes["dot_size"] = 40 + 760 * top_nodes["relative_centrality"]

# Plot
fig, ax = plt.subplots(figsize=(12, 9))

# Plot AOI boundary
gdf_states.to_crs('EPSG:4326').boundary.plot(ax=ax, color='black', linewidth=2, zorder=5, label='AOI')

ax.scatter(
    top_nodes.geometry.x,
    top_nodes.geometry.y,
    s=top_nodes["dot_size"],
    c="crimson",
    alpha=0.65,
    edgecolors="black",
    linewidths=0.5,
    zorder=6
)

# Size legend: marker radius keyed to node degree
if len(top_nodes) > 0:
    degree_vals = np.quantile(top_nodes["degree"], [0.25, 0.50, 0.90]).astype(int)
    degree_vals = np.unique(np.clip(degree_vals, 1, None))
    legend_handles = []
    for d in degree_vals:
        s = 40 + 760 * (d / max_degree)
        legend_handles.append(
            plt.scatter([], [], s=s, c="crimson", alpha=0.65, edgecolors="black", linewidths=0.5, label=f"Degree {d}")
        )
    size_legend = ax.legend(handles=legend_handles, title="Dot size by degree", loc="lower left", frameon=True)
    ax.add_artist(size_legend)

ax.set_title("Highest-Degree Grid Nodes (Dot Size = Relative Centrality)", fontsize=14, fontweight="bold")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_xlim(conus_bounds["lon_min"], conus_bounds["lon_max"])
ax.set_ylim(conus_bounds["lat_min"], conus_bounds["lat_max"])
ax.grid(True, alpha=0.25, linestyle="--")

print(f"Plotted {len(top_nodes)} highest-degree nodes (out of {len(nodes_plot)} CONUS nodes)")
print(f"Relative centrality range: {top_nodes['relative_centrality'].min():.3f} to {top_nodes['relative_centrality'].max():.3f}")

plt.tight_layout()
plt.show()